In [1]:
# Using the API

import os
from steam_web_api import Steam
import pandas as pd
from datetime import date

KEY = os.environ.get("MY_API_KEY")
steam = Steam(KEY)

In [2]:
# How to get the first game by STEAM ID

df = pd.DataFrame(columns=['STEAM ID', 'NAME', 'REQUIRED AGE', 'PRICE', 'HAS DISCOUNT', 'DISCOUNT', 'TODAY DATE'])
first_game = steam.apps.get_app_details(app_id=10)
first_game_det = first_game['10']['data']
first_steam_id = first_game_det['steam_appid']
j = 0
for i in steam.apps.search_games(first_game_det['name'], fetch_discounts=True)['apps']:
    if i['id'][0] == first_steam_id:
        try:
            has_discount = i['has_discount']
            discount = i['discount']
            j = float(i['price'][1:])
        except:
            has_discount = i['has_discount']
            discount = i['discount']
            j = i['price']
today_str = date.today().isoformat()
row_data = [
    first_steam_id,
    first_game_det["name"],
    first_game_det["required_age"],
    j,
    has_discount,
    discount,
    today_str,
]
df.loc[len(df)] = row_data
df.index = df.index + 1
df.to_json("steam_game.json", orient="records", indent=4, force_ascii=False)
df

https://store.steampowered.com/search/suggest?term=Counter-Strike&f=games&cc=US&realm=1&l=english


,STEAM ID,NAME,REQUIRED AGE,PRICE,HAS DISCOUNT,DISCOUNT,TODAY DATE
1,10,Counter-Strike,0,9.99,False,None,2026-09-24


In [3]:
import shutil

orig = 'steam_game.json'
copy = 'steam_games.json'

from pathlib import Path

file = Path('steam_games.json')
if file.is_file():
    pass
else:
    shutil.copy2(orig, copy)

In [ ]:
# Creating a list of the games by its STEAM ID and execute code several times changing STEAM ID's to getting news ID's

try:
    values = input(
        "Write two STEAM ID's and separate them by space, the difference between them have to be 1000 or less than 1000, write smallest first, example: '10000 11000', then execute the code again and write different ID's: "
    ).split()

    if len(values) != 2:
        raise ValueError(
            f"Expected exactly 2 values, but received {len(values)}."
        )

    first, last = map(int, values)

except ValueError as e:
    raise ValueError(f"Error in the input: {e}")

last = last + 1

if last < first:
    raise ValueError("Error: Please, write smallest first.")

if last < 0:
    raise ValueError("Error: Please, don't write negative numbers.")

if first < 0:
    raise ValueError("Error: Please, don't write negative numbers.")

if last - first > 1001:
    raise ValueError("Error: Difference between numbers is bigger than 1000.")

df = pd.read_json("steam_games.json")
df['STEAM ID'] = df['STEAM ID'].apply(lambda x: pd.to_datetime(x, unit="ms", errors="coerce") if isinstance(x, (int, float)) else pd.to_datetime(x, errors="coerce"))

for i in range (first, last, 10): # Specify first and last STEAM ID and iterate increasing by 10
    if i not in list(df['STEAM ID']):
        game = steam.apps.get_app_details(app_id=i)
    else:
        continue
    game_item = (game or {}).get(str(i)) # To avoid None values
    if game_item and game_item.get('success') and (game_item.get('data') or {}).get('type') == 'game': # using "{}" to get an empty dic to avoid None values
        game_det = game_item['data']
        steam_id = game_det['steam_appid']
        price = 0
        for j in steam.apps.search_games(game_det['name'], fetch_discounts=True)['apps']:
            if j['id'][0] == steam_id:
                try:
                    has_discount = j['has_discount']
                    discount = j['discount']
                    price = float(j['price'][1:])
                except:
                    has_discount = j['has_discount']
                    discount = j['discount']
                    price = j['price']
        today_str = date.today().isoformat()
        row_data = [
            steam_id,
            game_det["name"],
            game_det["required_age"],
            price,
            has_discount,
            discount,
            today_str,
        ]
        df.loc[-1] = row_data
        df.index = df.index + 1
        df = df.sort_index()
    else:
        continue
df = df.drop_duplicates()
df.to_json("steam_games.json", orient="records", indent=4, force_ascii=False)
df

ValueError: non convertible value 2026-09-22 with the unit 'ms', at position 0